# Занятие 6, пара 1. Сегментация


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Разделить картинку на области — ещё не значит понять объекты

В бинаризации мы отвечали: какие пиксели проходят условие? Сегментация назначает пикселям
области или классы. Можно отделить передний план от фона, выделить материал или получить
отдельную маску каждого объекта. Это разные постановки, даже если на экране похожие цветные пятна.

Сегодня сначала разберём несколько цветов, затем применим тот же принцип к фотографии чашки
400×600. На крупном изображении станет видно, что цветовые группы не обязаны совпадать с предметами.
После этого проверим маски и разберём два касающихся объекта.
Нужны RGB, маски и расстояние между векторами. Основной путь — NumPy/Matplotlib, без OpenCV.
Результат — сегментация фотографии, обоснованный выбор маски и объяснение того, что именно проверяет метрика.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Крупный пример, к которому вернёмся

`np.load` читает массив из комплекта. Это реальная фотография Rachel Michetti, CC0;
происхождение и условия всех файлов описаны в `segmentation/SOURCE.md`.
Для численных опытов переведём яркости из 0…255 в 0…1. Для расчёта возьмём каждый второй
пиксель: поле 200×300 всё ещё хорошо показывает структуру, но вычисляется быстрее на CPU.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

photo_u8 = np.load(DATA_DIR / 'segmentation' / 'coffee.npy',allow_pickle=False)
photo = photo_u8.astype(float) / 255
image = photo[::2,::2].copy()
plt.figure(figsize=(12,8))
plt.imshow(photo)
plt.title('Исходная фотография: что именно считать объектом?')
plt.axis('off')
plt.show()


**До кода:** отделится ли белая чашка от белой пены, если учитывать только цвет?
Обозначьте пальцем две разные части кадра с похожими цветами. Это будущий контрпример.

### 2. Цвет как точка и ближайший центр

RGB — вектор из трёх чисел. В k-means каждый пиксель относят к ближайшему центру, затем
центр пересчитывают как средний цвет назначенных пикселей. Повторяют до стабилизации.
Малое цветовое расстояние здесь не означает близость мест на изображении.
Сначала 4 пикселя и 2 центра; разность векторов, квадрат и сумма уже знакомы из дескрипторов.

`argmin(..., axis=1)` даст номер наименьшего расстояния в каждой строке. Корень можно не вычислять: порядок расстояний от этого не меняется. Таблица distances имеет четыре строки-пикселя и два столбца-центра. Например, в первой строке сравниваем близость одного красного пикселя к красному и синему центру, а не к соседним пикселям.


In [ ]:
pixels_small = np.array([[.9,.1,.1],[.8,.2,.1],[.1,.2,.8],[.1,.3,.9]])
centers_small = np.array([[1.,0.,0.],[0.,0.,1.]])
distances = np.zeros((len(pixels_small),len(centers_small)))
for j,center in enumerate(centers_small):
    difference = pixels_small-center
    distances[:,j] = np.sum(difference**2,axis=1)
print(distances)
labels_small = np.argmin(distances,axis=1)
print('Номера ближайших центров:',labels_small)


Теперь для каждого номера выбираем назначенные пиксели и считаем среднее по строкам.
Если группа пуста, сохраняем прежний центр; это простое учебное правило, а не единственный вариант.


In [ ]:
new_centers = centers_small.copy()
for j in range(len(centers_small)):
    members = pixels_small[labels_small==j]
    if len(members)>0:
        new_centers[j] = members.mean(axis=0)
print(new_centers)


### 3. Соберём повторение разобранных шагов

Одна итерация: назначить ближайший центр → пересчитать средние. `np.allclose` замечает,
что центры почти перестали меняться. Начальные цвета случайно выбираем из изображения,
фиксированный seed повторяет опыт. Лимит 30 итераций ограничивает время; это не гарантия
глобально лучшего результата. После последнего обновления ещё раз назначаем метки конечным центрам.
Функция ниже содержит именно эти шаги.

`rng.choice(..., replace=False)` выбирает разные индексы начальных пикселей без повторений. Это ещё не «правильные» центры: алгоритм затем передвинет их. У метода два разных параметра: K отвечает за число цветовых групп, max_steps — за предел числа повторений. Увеличение второго не заставит метод учитывать форму чашки.


In [ ]:
def assign_colors(pixels,centers):
    distances = np.zeros((len(pixels),len(centers)))
    for j,center in enumerate(centers):
        distances[:,j] = np.sum((pixels-center)**2,axis=1)
    return np.argmin(distances,axis=1)

def color_kmeans(pixels,k,seed=7,max_steps=30):
    rng = np.random.default_rng(seed)
    centers = pixels[rng.choice(len(pixels),size=k,replace=False)].copy()
    for step in range(max_steps):
        labels = assign_colors(pixels,centers)
        updated = centers.copy()
        for j in range(k):
            members = pixels[labels==j]
            if len(members)>0:
                updated[j] = members.mean(axis=0)
        converged = np.allclose(centers,updated,atol=1e-5)
        centers = updated
        if converged:
            break
    labels = assign_colors(pixels,centers)
    return centers,labels


Развернём изображение в строки RGB через `reshape(-1,3)`; −1 означает автоматически подобрать
число строк. Вернём меткам высоту и ширину. `centers[labels]` заменяет цвет каждого пикселя
цветом его группы. Это уменьшение палитры, не готовая семантическая разметка.


In [ ]:
pixels = image.reshape(-1,3)
centers,flat_labels = color_kmeans(pixels,4)
labels = flat_labels.reshape(image.shape[:2])
quantized = centers[labels]
fig,axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(image)
axes[0].set_title('Фотография')
axes[1].imshow(labels,cmap='tab10',vmin=0,vmax=9)
axes[1].set_title('Номера цветовых групп')
axes[2].imshow(quantized)
axes[2].set_title('Четыре средних цвета')
for axis in axes:
    axis.axis('off')
plt.show()


Номера 0…3 не означают «стол,чашка,пена,кофе»: при другой инициализации номера могут
переставиться, а области разбиться иначе. K задаёт число центров, но не число найденных предметов.
Некоторые группы могут оказаться пустыми. Похожий цвет в разных местах попадает в одну группу.

Посмотрим на все маски отдельно. `labels==j` оставляет пиксели группы j. На фото нет эталонной
маски чашки: мы обсуждаем ошибки качественно и не придумываем процент точности.


In [ ]:
fig,axes = plt.subplots(1,4,figsize=(16,4))
for j,axis in enumerate(axes):
    axis.imshow(labels==j,cmap='gray',vmin=0,vmax=1)
    axis.set_title('Цветовая группа '+str(j))
    axis.axis('off')
plt.show()


### ✏️ Ваш ход — заметить противоречие в указаниях

Человек отметил одну точку **объекта** и одну точку **фона**. Хотим получить маску цветовой
группы объекта. Напишите `my_select(label_map, object_point, background_point)`, где каждая точка —
пара `(y, x)`. Если указанные точки оказались в разных цветовых группах, верните bool-маску
группы объекта. Если в одной — верните `None`: по этим меткам одновременно выполнить оба
указания невозможно. Не придумывайте ложное уверенное выделение.

Проверьте на карте `[[2,2,7],[2,7,7]]`: объект `(0,0)`, фон `(0,2)`; затем фон `(0,1)`.
Перенумеруйте группы и убедитесь, что смысл ответа не изменился. Объясните, почему при конфликте
нужен другой признак или дополнительная информация, а не просто другой номер группы.
Дополнительно найдите на фотографии точку чашки и точку стола с одной цветовой меткой.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.1.1', my_select)


**Мой вывод:** …


### 4. Маска должна отвечать задаче

Нужна проверка, которая замечает и потерянный объект, и лишний фон.
**IoU** — площадь пересечения предсказанной маски с эталоном, делённая на площадь объединения.
Маски имеют одинаковую геометрию; True означает искомый передний план в обеих.
`&` даёт пересечение, `|` — объединение, `count_nonzero` считает True.
Для двух пустых масок здесь договоримся возвращать 1; это соглашение надо указывать.


In [ ]:
def mask_iou(prediction,truth):
    intersection = np.count_nonzero(prediction & truth)
    union = np.count_nonzero(prediction | truth)
    if union==0:
        return 1.0
    return intersection/union

truth_small = np.array([[0,1,1],[0,1,0]],dtype=bool)
pred_small = np.array([[0,1,1],[0,0,1]],dtype=bool)
print('Пересечение:',np.count_nonzero(pred_small & truth_small))
print('Объединение:',np.count_nonzero(pred_small | truth_small))
print('IoU:',mask_iou(pred_small,truth_small))


### 5. Большой контрпример: почти весь кадр — фон

Сделаем синтетический кадр 240×360 с маленьким объектом. Эталон известен из способа построения.
Первый метод всегда говорит «фон». Второй находит объект, но оставляет вокруг него рамку.
На общей доле правильных пикселей первый может выглядеть великолепно. Рассмотрим маски крупно,
а не только процент. `np.indices` создаёт координаты y и x для всей сетки.


In [ ]:
y,x = np.indices((240,360))
truth = (x>=140)&(x<200)&(y>=90)&(y<150)
empty_prediction = np.zeros_like(truth)
wide_prediction = (x>=135)&(x<205)&(y>=85)&(y<155)
fig,axes = plt.subplots(1,3,figsize=(14,5))
for axis,picture,title in zip(axes,[truth,empty_prediction,wide_prediction],['Эталон','Всегда фон','Объект с лишней рамкой']):
    axis.imshow(picture,cmap='gray',vmin=0,vmax=1)
    axis.set_title(title)
    axis.axis('off')
plt.show()
print('Доля правильных у всегда-фона:',np.mean(empty_prediction==truth))
print('IoU всегда-фона:',mask_iou(empty_prediction,truth))


### ✏️ Ваш ход — выбрать результат по цене ошибки

Есть два результата: `wide_prediction` и `narrow_prediction`,
где узкая маска занимает x=145…194,y=95…144. Сравните IoU, число потерянных пикселей объекта
(truth & ~prediction) и число лишних пикселей фона (prediction & ~truth).
Сохраните словарь `my_metrics`: ключи wide/narrow, значения(число_пропусков,число_лишних,IoU).
Для вырезания объекта важнее не отрезать его часть, а для точного измерения площади важны обе ошибки.
Выберите результат для каждой задачи и обоснуйте компромисс. Одного имени «лучшей маски» недостаточно.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.1.2', my_metrics)


**Мой вывод:** …


### 6. Один передний план или два объекта?

В объединённой маске два соприкасающихся круга выглядят одной областью. Для задачи «найти материал»
это допустимо. Для задачи «посчитать детали» нужна отдельная метка каждого экземпляра.
**Маркеры** — указания, где начинается каждый объект. Watershed можно представить как рост
областей от маркеров по рельефу; встречи областей образуют разделяющую границу.

Ниже **заранее подготовленная иллюстрация**, а не запуск watershed на вашем компьютере:
рельеф получен из расстояния до фона, две начальные точки предоставлены. Подробности в SOURCE.md.
Удачный результат зависит от маркеров и рельефа: число маркеров не возникает из знания алгоритмом слова «монета».

Почему рельеф связан с расстоянием? Внутри каждого круга пиксели центра дальше всего от фона. Если поменять знак расстояния, центры станут «впадинами». Две предоставленные точки начинают заполнять эти впадины; между ними появляется граница. Если поставить всего один маркер, сам этот опыт не научится угадывать, что мы задумали два предмета. Это полезная подсказка человеку, а не магическое распознавание объектов.


In [ ]:
example = np.load(DATA_DIR / 'segmentation' / 'touching.npz',allow_pickle=False)
union_mask = example['mask']
marker_map = example['markers']
instance_labels = example['reference']
fig,axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(union_mask,cmap='gray')
axes[0].set_title('Общий передний план')
axes[1].imshow(union_mask,cmap='gray')
marker_y,marker_x = np.nonzero(marker_map)
axes[1].scatter(marker_x,marker_y,c=['red','cyan'],s=70)
axes[1].set_title('Две предоставленные точки')
axes[2].imshow(instance_labels,cmap='tab10',vmin=0,vmax=9)
axes[2].set_title('Готовая иллюстрация watershed')
for axis in axes:
    axis.axis('off')
plt.show()


### ✏️ Ваш ход — проверить, что метрика вообще видит ошибку

Первый алгоритм возвращает только `union_mask`;
второй — отдельные `instance_labels`. Получите их семантический IoU в `my_semantic_score`
(сравните передний план с передним планом). Найдите число ненулевых меток второго результата
в `my_instance_count`, используя np.unique: эта функция возвращает различные значения массива.
Может ли идеальный IoU первого результата подтвердить правильный подсчёт двух объектов?
Запишите недостающую проверку словами. Не называйте подключение готового результата своим запуском watershed.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.1.3', my_semantic_score, my_instance_count)


**Мой вывод:** …


### Резерв: что случится с реальными монетами?

На фото монет из Brooklyn Museum яркости фона и объектов перекрываются.
Покажем порог 120: где он режет монеты, а где захватывает фон? Порог не распознаёт монету,
он проверяет число. Попробуйте другой порог и объясните, можно ли исправить обе ошибки одновременно.
Эталонной маски к фото не выдано; IoU «на глаз» не назначаем.


In [ ]:
coins = np.load(DATA_DIR / 'segmentation' / 'coins.npy',allow_pickle=False)
fig,axes = plt.subplots(1,2,figsize=(12,6))
axes[0].imshow(coins,cmap='gray',vmin=0,vmax=255)
axes[0].set_title('Реальная фотография монет')
axes[1].imshow(coins>120,cmap='gray')
axes[1].set_title('Порог120: найдите обе ошибки')
for axis in axes:
    axis.axis('off')
plt.show()


**Итог:** цветовая группа, передний план и отдельный предмет — разные результаты.
Показываем крупный результат, смотрим ошибки, выбираем метрику под вопрос.
Сохраните свою тетрадь с выводами, GitHub не обязателен.

По желанию: [watershed с маркерами](https://docs.opencv.org/4.x/d3/db4/tutorial_py_watershed.html),
[GrabCut с указанием объекта человеком](https://docs.opencv.org/4.x/d8/d83/tutorial_py_grabcut.html).
GrabCut использует цветовую модель и связи соседних пикселей; прямоугольник задаёт исходное
предположение, а не готовую маску. Спектральную/иерархическую кластеризацию всей большой картинки
в обязательный CPU-маршрут не включаем.
